Este script tem como finalidade limpar  e padronizar o csv para que funcione com diferentes bairros apenas com o nome

In [7]:
import pandas as pd
import numpy as np

In [8]:
# DELETANDO LINHAS E COLUNAS DESNECESSÁRIAS
def del_lines(df):
    df = df.drop(index = [0,1,2])
    df = df.drop(columns=['SECRETARIA MUNICIPAL DA SAUDE DE ARARANGUA - SC', "Unnamed: 0", 'Unnamed: 3','Unnamed: 5', 'Unnamed: 6','Unnamed: 8', 'Unnamed: 12', 'Unnamed: 13'])

    df = df.rename(columns={
        "Unnamed: 1": "Paciente",
        "Unnamed: 4" : "Vacina estratégia e dose",
        "Unnamed: 7": "Lote e Data de Validade",
        "Unnamed: 9" : "Data da aplicação",
        "Unnamed: 10" : "Qtde.",
        "Unnamed: 11" : "Origem"
    })

    df = df.dropna(axis=0, how="all")

    df = df.dropna(axis=0, thresh=2)
    df = df[(df['Paciente'] != 'Paciente') & 
            (df['Vacina estratégia e dose'] != 'Vacina, Estratégia e Dose') & 
            (df['Lote e Data de Validade'] != 'Lote e Data de Validade') & 
            (df['Data da aplicação'] != 'Aplicado') & 
            (df['Qtde.'] != 'Qtde') & 
            (df['Origem'] != 'Origem')
            ]

    return df

In [9]:
# SEPARANDO COLUNAS
def sep_columns(df):
    df = df.reset_index(drop=True)
    df[["Nome da vacina", "Estratégia", "Dose"]] = df["Vacina estratégia e dose"].str.split("-", n = 2, expand=True)
    df[["Lote", "Validade do lote"]] = df["Lote e Data de Validade"].str.split("-", n = 1, expand=True)

    df["Código da vacina"] = df["Nome da vacina"].str.extract(r"\((.*?)\)")
    df["Código da estratégia"] = df["Estratégia"].str.extract(r"\((.*?)\)")
    df["Código da dose"] = df["Dose"].str.extract(r"\((.*?)\)")

    df["Nome da vacina"] = df["Nome da vacina"].str.replace(r"\(.*?\)", "", regex=True).str.strip()
    df["Estratégia"] = df["Estratégia"].str.replace(r"\(.*?\)", "", regex=True).str.strip()
    df["Dose"] = df["Dose"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

    return df

In [10]:
def ajustar_data(data):
    if pd.isna(data):
        return data 
    data = str(data)
    if len(data.split('/')) == 3:
        return data
    else:
        return "01/" + data

def ajust_type(df):
    numeric_coluns = ['Código da vacina', 'Código da estratégia', 'Código da dose']
    date_coluns = ['Validade do lote', 'Data da aplicação']

    for colunm in numeric_coluns:
        df[colunm] = pd.to_numeric(df[colunm], errors="coerce").astype("Int64")

    for colunm in date_coluns:
        df[colunm] = df[colunm].apply(ajustar_data)

    return df

In [11]:
def limpar_csv(NOME_BAIRRO):
    NOME_BAIRRO_CAMINHO = NOME_BAIRRO.replace(" ", "_")

    NOMES_COLUNAS = [
        'Nome do bairro',
        'Nome da vacina',
        'Código da vacina',
        'Estratégia',
        'Código da estratégia',
        'Dose',
        'Código da dose',
        'Vacina estratégia e dose',
        'Lote',
        'Validade do lote',
        'Data da aplicação',
        'Origem'
    ]

    CAMINHO_ENTRADA = f'../{NOME_BAIRRO}/Vacinas_Aplicadas_{NOME_BAIRRO_CAMINHO}_Anonimizadas.csv'
    CAMINHO_SAIDA = f'../{NOME_BAIRRO}/Vacinas_Aplicadas_{NOME_BAIRRO_CAMINHO}_Processado.csv'

    df = pd.read_csv(CAMINHO_ENTRADA, encoding="utf-8")

    df = del_lines(df)
    df = sep_columns(df)
    df = ajust_type(df)

    #Adicionando o bairro e reordenando
    df['Nome do bairro'] = NOME_BAIRRO
    df = df[NOMES_COLUNAS]

    df.info()
    df.head()

    df.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8")

In [12]:
limpar_csv("Morro dos Conventos")
limpar_csv("Urussanguinha")

<class 'pandas.DataFrame'>
RangeIndex: 422 entries, 0 to 421
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Nome do bairro            422 non-null    str  
 1   Nome da vacina            422 non-null    str  
 2   Código da vacina          398 non-null    Int64
 3   Estratégia                422 non-null    str  
 4   Código da estratégia      417 non-null    Int64
 5   Dose                      422 non-null    str  
 6   Código da dose            417 non-null    Int64
 7   Vacina estratégia e dose  422 non-null    str  
 8   Lote                      422 non-null    str  
 9   Validade do lote          422 non-null    str  
 10  Data da aplicação         422 non-null    str  
 11  Origem                    109 non-null    str  
dtypes: Int64(3), str(9)
memory usage: 40.9 KB
<class 'pandas.DataFrame'>
RangeIndex: 832 entries, 0 to 831
Data columns (total 12 columns):
 #   Column            